In [1]:
!pip install transformers torch scikit-learn pandas numpy

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import torch

print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

2.14.0+cpu
GPU available: False


In [4]:
import os

print("Current folder:")
print(os.getcwd())

print("\nData folder exists:")
print(os.path.exists("data"))

print("\nProcessed folder exists:")
print(os.path.exists("data/processed"))

print("\nFiles in data/processed:")
if os.path.exists("data/processed"):
    print(os.listdir("data/processed"))

Current folder:
d:\Explainable-AI-News-Intelligence-System\notebooks

Data folder exists:
False

Processed folder exists:
False

Files in data/processed:


In [6]:
import os
import glob
import pandas as pd

# Find news_data.csv anywhere inside the project
files = glob.glob(
    "**/news_data.csv",
    recursive=True
)

print("Found files:")
print(files)

Found files:
[]


In [7]:
import pandas as pd
import re
import os

# Absolute paths to your existing datasets
fake_path = r"D:\Explainable-AI-News-Intelligence-System\data\Fake.csv"
true_path = r"D:\Explainable-AI-News-Intelligence-System\data\True.csv"

# Load datasets
fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)

print("Fake:", fake_df.shape)
print("True:", true_df.shape)

# IMPORTANT: keep the same labels as your existing project
fake_df["label"] = 1
true_df["label"] = 0

# Combine
df = pd.concat(
    [fake_df, true_df],
    ignore_index=True
)

print("Combined:", df.shape)
print(df["label"].value_counts())

Fake: (23481, 4)
True: (21417, 4)
Combined: (44898, 5)
label
1    23481
0    21417
Name: count, dtype: int64


In [8]:
def clean_text(text):
    text = str(text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Handle missing values
df = df.dropna(subset=["text"])

df["title"] = df["title"].fillna("")
df["subject"] = df["subject"].fillna("Unknown")
df["date"] = df["date"].fillna("Unknown")

# Combine title + article
df["combined_text"] = (
    df["title"] + " " + df["text"]
)

# Clean
df["cleaned_text"] = (
    df["combined_text"].apply(clean_text)
)

# Remove empty text
df = df[
    df["cleaned_text"].str.strip() != ""
]

print("Final shape:", df.shape)
print(df[["cleaned_text", "label"]].head())
print("\nLabels:")
print(df["label"].value_counts())

Final shape: (44889, 7)
                                        cleaned_text  label
0  donald trump sends out embarrassing new years ...      1
1  drunk bragging trump staffer started russian c...      1
2  sheriff david clarke becomes an internet joke ...      1
3  trump is so obsessed he even has obamas name c...      1
4  pope francis just called out donald trump duri...      1

Labels:
label
1    23472
0    21417
Name: count, dtype: int64


In [9]:
df_bert = df[
    ["cleaned_text", "label"]
].sample(
    n=min(10000, len(df)),
    random_state=42
).reset_index(drop=True)

print("BERT dataset:", df_bert.shape)
print(df_bert["label"].value_counts())

BERT dataset: (10000, 2)
label
1    5271
0    4729
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

X = df_bert["cleaned_text"]
y = df_bert["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 8000
Test: 2000


In [11]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer loaded successfully")

c:\Users\HP\.conda\envs\newsbert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HP\.conda\envs\newsbert\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, se

Tokenizer loaded successfully


In [12]:
sample_text = X_train.iloc[0]

tokens = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("Input IDs shape:", tokens["input_ids"].shape)
print("Attention mask shape:", tokens["attention_mask"].shape)

Input IDs shape: torch.Size([1, 128])
Attention mask shape: torch.Size([1, 128])


In [13]:
import torch
from torch.utils.data import Dataset

class NewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

In [14]:
train_dataset = NewsDataset(
    X_train,
    y_train,
    tokenizer
)

test_dataset = NewsDataset(
    X_test,
    y_test,
    tokenizer
)

print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 8000
Test dataset: 2000


In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 1000
Testing batches: 250


In [16]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("BERT model loaded successfully")

c:\Users\HP\.conda\envs\newsbert\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 606.63it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased

BERT model loaded successfully


In [17]:
from torch.optim import AdamW
import torch

device = torch.device("cpu")

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Device:", device)

Device: cpu


In [18]:
from tqdm.auto import tqdm

# Small test: 500 training samples
small_train_dataset = NewsDataset(
    X_train.iloc[:500],
    y_train.iloc[:500],
    tokenizer
)

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=8,
    shuffle=True
)

model.train()

total_loss = 0

for batch in tqdm(small_train_loader, desc="BERT training test"):

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

print("Training test completed!")
print("Average loss:", total_loss / len(small_train_loader))

BERT training test: 100%|██████████| 63/63 [05:01<00:00,  4.78s/it]

Training test completed!
Average loss: 0.2745172729390481


In [19]:
df_bert_final = df[
    ["cleaned_text", "label"]
].sample(
    n=2000,
    random_state=42
).reset_index(drop=True)

X = df_bert_final["cleaned_text"]
y = df_bert_final["label"]

X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train_bert))
print("Test:", len(X_test_bert))

Train: 1600
Test: 400


In [20]:
train_dataset_bert = NewsDataset(
    X_train_bert,
    y_train_bert,
    tokenizer
)

test_dataset_bert = NewsDataset(
    X_test_bert,
    y_test_bert,
    tokenizer
)

train_loader_bert = DataLoader(
    train_dataset_bert,
    batch_size=8,
    shuffle=True
)

test_loader_bert = DataLoader(
    test_dataset_bert,
    batch_size=8,
    shuffle=False
)

print("Training batches:", len(train_loader_bert))
print("Testing batches:", len(test_loader_bert))

Training batches: 200
Testing batches: 50


In [21]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW
import torch

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

device = torch.device("cpu")
model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Fresh BERT ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 369.83it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Fresh BERT ready


In [22]:
from tqdm.auto import tqdm

model.train()

total_loss = 0

for batch in tqdm(
    train_loader_bert,
    desc="BERT Training"
):

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

print("Training completed!")
print("Average loss:", total_loss / len(train_loader_bert))

BERT Training: 100%|██████████| 200/200 [17:57<00:00,  5.39s/it]

Training completed!
Average loss: 0.14033315556123852


In [23]:
from sklearn.model_selection import train_test_split

# Use the complete dataset
df_final = df[
    ["cleaned_text", "label"]
].dropna().reset_index(drop=True)

# Remove empty text
df_final = df_final[
    df_final["cleaned_text"].str.strip() != ""
].reset_index(drop=True)

print("Total articles:", len(df_final))
print("\nLabel distribution:")
print(df_final["label"].value_counts())

# 80/20 split
X_full = df_final["cleaned_text"]
y_full = df_final["label"]

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=42,
    stratify=y_full
)

print("\nTraining articles:", len(X_train_full))
print("Testing articles:", len(X_test_full))

Total articles: 44889

Label distribution:
label
1    23472
0    21417
Name: count, dtype: int64

Training articles: 35911
Testing articles: 8978


In [24]:
train_dataset_full = NewsDataset(
    X_train_full,
    y_train_full,
    tokenizer,
    max_length=128
)

test_dataset_full = NewsDataset(
    X_test_full,
    y_test_full,
    tokenizer,
    max_length=128
)

print("Training dataset:", len(train_dataset_full))
print("Testing dataset:", len(test_dataset_full))

Training dataset: 35911
Testing dataset: 8978


In [25]:
from torch.utils.data import DataLoader

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

test_loader_full = DataLoader(
    test_dataset_full,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader_full))
print("Testing batches:", len(test_loader_full))

Training batches: 4489
Testing batches: 1123


In [26]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW
import torch

# Fresh pretrained BERT
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

device = torch.device("cpu")

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Fresh BERT model ready")
print("Device:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 292.30it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Fresh BERT model ready
Device: cpu


In [28]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [29]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("BERT tokenizer ready")

BERT tokenizer ready


In [30]:
train_dataset_full = NewsDataset(
    X_train_full,
    y_train_full,
    tokenizer,
    max_length=128
)

test_dataset_full = NewsDataset(
    X_test_full,
    y_test_full,
    tokenizer,
    max_length=128
)

In [31]:
from torch.utils.data import DataLoader

train_loader_full = DataLoader(
    train_dataset_full,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

test_loader_full = DataLoader(
    test_dataset_full,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader_full))
print("Testing batches:", len(test_loader_full))

Training batches: 4489
Testing batches: 1123


In [32]:
from transformers import BertForSequenceClassification
import torch

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

device = torch.device("cpu")
model = model.to(device)

print("Fresh BERT loaded")
print("Device:", device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 607.62it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Fresh BERT loaded
Device: cpu


In [33]:
# Freeze the pretrained BERT encoder
for param in model.bert.parameters():
    param.requires_grad = False

print("BERT encoder frozen")

BERT encoder frozen


In [34]:
# Check how many parameters will actually be trained

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable parameters:", trainable)
print("Total parameters:", total)

Trainable parameters: 1538
Total parameters: 109483778


In [35]:
from torch.optim import AdamW

optimizer = AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-5
)

print("Optimizer ready")

Optimizer ready


In [36]:
import time

model.train()

start_time = time.time()

for i, batch in enumerate(train_loader_full):
    if i >= 20:
        break

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

elapsed = time.time() - start_time

print(f"20 batches completed in {elapsed:.2f} seconds")
print(f"Average time per batch: {elapsed/20:.2f} seconds")

20 batches completed in 32.12 seconds
Average time per batch: 1.61 seconds


In [37]:
from transformers import BertForSequenceClassification

# Fresh BERT model for final training
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model = model.to(device)

# Freeze BERT encoder
for param in model.bert.parameters():
    param.requires_grad = False

print("Fresh BERT model ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 558.12it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Fresh BERT model ready


In [38]:
from torch.optim import AdamW

optimizer = AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=2e-5
)

print("Final optimizer ready")

Final optimizer ready


In [39]:
from tqdm.auto import tqdm
import time

model.train()

num_epochs = 1

for epoch in range(num_epochs):

    total_loss = 0
    start_time = time.time()

    progress_bar = tqdm(
        train_loader_full,
        desc=f"BERT Training Epoch {epoch+1}/{num_epochs}"
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=loss.item()
        )

    average_loss = total_loss / len(train_loader_full)

    elapsed = time.time() - start_time

    print(f"\nEpoch {epoch+1} completed")
    print(f"Average Loss: {average_loss:.4f}")
    print(f"Training Time: {elapsed/60:.2f} minutes")

BERT Training Epoch 1/1: 100%|██████████| 4489/4489 [3:48:40<00:00,  3.06s/it, loss=0.587]   


Epoch 1 completed
Average Loss: 0.6147
Training Time: 228.68 minutes


In [40]:
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="BERT Evaluation"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# Metrics
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print("BERT RESULTS")
print("=" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Class 0", "Class 1"]
    )
)

BERT Evaluation: 100%|██████████| 250/250 [06:01<00:00,  1.44s/it]


BERT RESULTS
Accuracy : 0.8650
Precision: 0.8506
Recall   : 0.9023
F1 Score : 0.8757

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.88      0.82      0.85       946
     Class 1       0.85      0.90      0.88      1054

    accuracy                           0.86      2000
   macro avg       0.87      0.86      0.86      2000
weighted avg       0.87      0.86      0.86      2000



In [41]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[779 167]
 [103 951]]


In [42]:
import os
import torch

os.makedirs("../models", exist_ok=True)

torch.save(
    model.state_dict(),
    "../models/bert_news_classifier.pt"
)

print("BERT model saved successfully")

BERT model saved successfully


In [43]:
print("=" * 60)
print("BERT MODEL FINAL RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

BERT MODEL FINAL RESULTS
Accuracy : 0.8650
Precision: 0.8506
Recall   : 0.9023
F1 Score : 0.8757

Confusion Matrix:
[[779 167]
 [103 951]]


In [44]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "TF-IDF + Random Forest",
        "BERT"
    ],
    "Accuracy": [
        0.9897,
        accuracy
    ],
    "Precision": [
        0.9875,
        precision
    ],
    "Recall": [
        0.9908,
        recall
    ],
    "F1 Score": [
        0.9892,
        f1
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,TF-IDF + Random Forest,0.9897,0.987500,0.990800,0.989200
1,BERT,0.8650,0.850626,0.902277,0.875691
